This notebook loads the pickled results produced by `scaling_effdim_decexp.py` and plots how the normalized effective dimension (ED, Eq. 12) scales with the correlation-spectrum purity $\mathrm{tr}\,S^4$, obtained by varying the decay exponent `dec_exp` of the correlation spectrum S, for a few fixed architectures (no. of parameters M, local per-parameter basis dimension Dloc, input-basis dimension D).

Import the plotting and numerical packages, and add the `useful_functions` folder to the module search path so the tensor-network, FIM, and model-construction helper modules used by the generating script can be (re)imported.

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import pickle
import numpy

import pennylane.numpy as np




import matplotlib.pyplot as plt


# Current path for importing custom functions
path_base = '/home/b/b309245/FIM_Training_Bias_RegressionModels/fourier_models_training_and_fim/'
sys.path.insert(0, path_base + 'useful_functions')

import model_constructor_functions
importlib.reload(model_constructor_functions)

import ortho_matrices_functions
importlib.reload(ortho_matrices_functions)

import tensor_network_functions_np
importlib.reload(tensor_network_functions_np)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import training_functions_jax
importlib.reload(training_functions_jax)

Set the results folder and the fixed parameters shared by all experiments (D, no. of parameter samples/realizations, M), then list, for each experiment/curve, the architecture (bond dimension $\chi$, Dloc, D, M) whose scan over the correlation-spectrum decay exponent (and hence $\mathrm{tr}\,S^4$) is to be loaded.

In [ ]:
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/scaling_effdim_TN_decay_spectrS/'

# Dimension of input function space
dim_in = 50
name_dim_in = str(dim_in)

# No. of random parameter samples for evaluating normalized eff. dim.
no_samples = 200
no_par_samples_name = str(no_samples)

# No. of random orthogonal matrix realizations per S decay exponent
no_matrix_realiz = 30
no_V_samples_name = str(no_matrix_realiz)

# No. of parameters
no_params = 60
name_no_params = str(no_params)

########################### Exps. to plot ###########################

# Vector of bond dimensions
bond_dim_vec = [100, 100, 120]
name_bond_dim_vec = [str(bond_dim) for bond_dim in bond_dim_vec]

# Local dimension of parameter functions space
local_dim_param_vec = [5, 5, 7]
name_dim_par_loc_vec = [str(local_dim_param) for local_dim_param in local_dim_param_vec]

# Dimension of input function space
dim_in_vec = [50, 80, 50]
name_dim_in_vec = [str(dim_in) for dim_in in dim_in_vec]

# Dimension of input function space
no_params_vec = [60, 40, 80]
name_no_params_vec = [str(no_params) for no_params in no_params_vec]

no_exps = len(local_dim_param_vec)

For each experiment (each fixed architecture), find and load all pickled `norm_eff_dim_*` files matching that architecture, concatenating the swept correlation-spectrum decay-exponent / purity ($\mathrm{tr}\,S^4$) values and normalized ED arrays across all random model realizations into a per-experiment dictionary `dict_exps[i]`.

In [ ]:
dict_exps = dict()

for i in range(no_exps):
    local_dim_param = local_dim_param_vec[i]
    name_dim_par_loc = name_dim_par_loc_vec[i]
    no_params = no_params_vec[i]
    name_no_params = name_no_params_vec[i]
    dim_in = dim_in_vec[i]
    name_dim_in = name_dim_in_vec[i]
    bond_dim = bond_dim_vec[i]
    name_bond_dim = name_bond_dim_vec[i]

    name_end_0 = ('_BondDim' + name_bond_dim + '_DimIn' + name_dim_in + '_DimLocPar' + name_dim_par_loc + 
                  '_Nparams' + name_no_params + '_NsamplesPar' + no_par_samples_name + 
                  '_NsamplesV' + no_V_samples_name + '_DecExp')
    
    namefileini = 'norm_eff_dim' + name_end_0
    listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(namefileini))]
    print(len(listallfiles))
    
    decay_exp_all = []
    purity_S_all = []
    norm_eff_dim_all = []
    
    for filename in listallfiles:
        path_file = os.path.join(results_folder, filename)
        with open(path_file, 'rb') as f:
            dict_norm_ed = pickle.load(f)
    
        decay_exp_vec = dict_norm_ed['decay_exp_all']
        purity_S_vec = dict_norm_ed['purity_S_all']
        norm_eff_dim_vec = dict_norm_ed['norm_eff_dim_all']
    
        decay_exp_all.append(decay_exp_vec)
        purity_S_all.append(purity_S_vec)
        norm_eff_dim_all.append(norm_eff_dim_vec)
    
    decay_exp_all = np.concatenate(decay_exp_all)
    purity_S_all = np.concatenate(purity_S_all)
    norm_eff_dim_all = np.concatenate(norm_eff_dim_all)

    dict_exp = dict()
    dict_exp['local_dim_param'] = local_dim_param
    dict_exp['no_params'] = no_params
    dict_exp['dim_in'] = dim_in
    dict_exp['bond_dim'] = bond_dim
    dict_exp['purity_S_all'] = purity_S_all
    dict_exp['decay_exp_all'] = decay_exp_all
    dict_exp['norm_eff_dim_all'] = norm_eff_dim_all
    dict_exps[i] = dict_exp

Plot the normalized effective dimension $\hat d_{\mathrm{eff}}$ directly against the correlation-spectrum purity $\mathrm{tr}\,S^4$, one curve per fixed (M, Dloc, D) architecture.

In [ ]:
fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']
inds = [0, 1, 2]
for ind in range(len(inds)):
    i = inds[ind]
    dict_exp = dict_exps[i]
    no_params = dict_exp['no_params']
    local_dim_param = dict_exp['local_dim_param']
    dim_in = dict_exp['dim_in']
    norm_eff_dim_all = dict_exp['norm_eff_dim_all']
    purity_S_all = numpy.asarray(dict_exp['purity_S_all'])
    lbl = r'$M=' + str(no_params) + ',\,' + r'\tilde{d}=' + str(local_dim_param) + ',\,D=' + str(dim_in) + '$'
    x = np.asarray(purity_S_all)
    y = np.asarray(norm_eff_dim_all)
    ax.plot(x, y, '.', color=clrs[ind], markersize=12, label=lbl)

#ax.legend(fontsize=22, loc='lower left')
ax.legend(fontsize=22)
ax.set_xlabel(r'$\mathrm{tr}\;S^4$', fontsize=fs)
ax.set_ylabel(r'$\hat{d}_{\mathrm{eff}}$', fontsize=fs)
ax.set_xscale('log')
#ax.set_yscale('log')
ax.set_ylim([0.3, 0.97])

fig.savefig('EffDimScaling_vs_trS4_TN__chi_100.png', bbox_inches='tight')
fig.savefig('EffDimScaling_vs_trS4_TN__chi_100.pdf', bbox_inches='tight')